# Notebook 40 – Feature Engineering & Modell III
**Projekt:** Wirtschaftlichkeit von Airbnb-Listings in Spanien  
**Autorin:** Jasmin Müller (951624)  
**Modell III:** Optimierter Decision Tree Regressor (Fallback: GradientBoostingRegressor)  

---
### Struktur dieses Notebooks
1. Daten laden (bereinigter Datensatz von Person 1)
2. Feature Engineering (eigenständig, modular)
3. Train/Test-Split
4. Modell III: Optimierter Decision Tree mit GridSearchCV
5. Evaluation (MAE, RMSE, R²)
6. Feature Importance speichern (für Notebook 50)
7. Modell speichern (für Notebook 50)

> **Hinweis zur Modularität:** Das Feature Engineering ist in einer eigenen Funktion `build_features()` gekapselt.
> Falls zu einem späteren Zeitpunkt der engineered Datensatz von Person 2 (Mareike) vorliegt,
> kann dieser direkt ab Abschnitt 3 (Train/Test-Split) eingehängt werden – ohne den Rest zu ändern.

## 0 – Imports & Konfiguration

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import os
import pickle
warnings.filterwarnings('ignore')

from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import GradientBoostingRegressor   # Fallback
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler

# ── Reproduzierbarkeit ──────────────────────────────────────────────────────
RANDOM_STATE = 42

# ── Pfade ───────────────────────────────────────────────────────────────────
# Passe diesen Pfad an deine GitHub-Struktur an:
DATA_PATH   = 'listings_spanien_cleaned.csv'   # Output von Person 1
CACHE_DIR   = 'cache/'
OUTPUT_DIR  = 'outputs/'

os.makedirs(CACHE_DIR,  exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

print('Setup abgeschlossen ✓')

KeyboardInterrupt: 

## 1 – Daten laden

In [ ]:
df_raw = pd.read_csv(DATA_PATH)
print(f'Datensatz geladen: {df_raw.shape[0]:,} Zeilen, {df_raw.shape[1]} Spalten')
df_raw.head(3)

## 2 – Feature Engineering

Die gesamte Logik ist in `build_features()` gekapselt.  
Das erleichtert den späteren Austausch gegen Mareikes Pipeline.

### Engineerte Features im Überblick

| Feature | Typ | Beschreibung |
|---|---|---|
| `room_type_*` | OHE | Zimmerart (Entire home, Private, Hotel, Shared) |
| `region_*` | OHE | Region (9 Regionen in Spanien) |
| `dist_beach` | numerisch | Euklidische Annäherung: Distanz zum nächsten Strand-Hotspot |
| `dist_city_center` | numerisch | Distanz zum jeweiligen Stadtzentrum der Region |
| `is_licensed` | binär | Hat das Listing eine gültige Lizenz (1/0) |
| `host_is_superhost_proxy` | binär | Host hat >1 Listing (Proxy für professionellen Host) |
| `min_nights_capped` | numerisch | minimum_nights gekappt bei 30 (reduziert Ausreißereinfluss) |
| `review_score_proxy` | numerisch | reviews_per_month × availability_365 (Aktivitäts-Score) |

In [ ]:
# ── Tourismus-Referenzpunkte (lat, lon) ─────────────────────────────────────
# Stadtzentren der 9 Regionen
CITY_CENTERS = {
    'Barcelona': (41.3851, 2.1734),
    'Madrid':    (40.4168, -3.7038),
    'Mallorca':  (39.5696, 2.6502),
    'Menorca':   (39.9496, 4.1156),
    'Girona':    (41.9794, 2.8214),
    'Malaga':    (36.7213, -4.4214),
    'Sevilla':   (37.3891, -5.9845),
    'Valencia':  (39.4699, -0.3763),
    'Euskadi':   (43.2630, -2.9350),   # Bilbao als Zentrum
}

# Bekannte Strand-/Tourismusmagnete (lat, lon)
BEACH_HOTSPOTS = [
    (39.5296, 2.3873),   # Magaluf, Mallorca
    (41.2754, 1.9862),   # Sitges, Barcelona
    (36.5007, -4.6679),  # Torremolinos, Malaga
    (39.8628, 4.2591),   # Platja de ses Salines, Menorca
    (41.9787, 3.2169),   # Roses, Girona
    (39.3560, -0.3319),  # Cullera, Valencia
]

def haversine_approx(lat1, lon1, lat2, lon2):
    """Schnelle euklidische Annäherung in Grad-Distanz (kein echter km-Wert,
    aber konsistent als relatives Feature). Für ML-Zwecke ausreichend."""
    return np.sqrt((lat1 - lat2)**2 + (lon1 - lon2)**2)


def build_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Eigenständiges Feature Engineering für Modell III.
    Input:  Bereinigter Rohdatensatz (Output von Person 1)
    Output: Modellbereiter DataFrame mit allen Features
    
    MODULARITÄTS-HINWEIS:
    Wenn Mareikes CSV (cache/20_features_engineered.csv) vorliegt,
    kann diese Funktion übersprungen werden – einfach ab Abschnitt 3
    mit df_feat = pd.read_csv('cache/20_features_engineered.csv') einsteigen.
    """
    df = df.copy()
    
    # ── 1. Zieldistanz zum Stadtzentrum der eigenen Region ──────────────────
    df['dist_city_center'] = df.apply(
        lambda row: haversine_approx(
            row['latitude'], row['longitude'],
            *CITY_CENTERS.get(row['region'], (row['latitude'], row['longitude']))
        ), axis=1
    )
    
    # ── 2. Distanz zum nächsten Strand-Hotspot ──────────────────────────────
    df['dist_beach'] = df.apply(
        lambda row: min(
            haversine_approx(row['latitude'], row['longitude'], blat, blon)
            for blat, blon in BEACH_HOTSPOTS
        ), axis=1
    )
    
    # ── 3. Lizenz-Flag ──────────────────────────────────────────────────────
    df['is_licensed'] = df['license'].notna().astype(int)
    
    # ── 4. Professioneller Host (Proxy) ─────────────────────────────────────
    df['host_is_pro'] = (df['calculated_host_listings_count'] > 1).astype(int)
    
    # ── 5. minimum_nights kappen ────────────────────────────────────────────
    df['min_nights_capped'] = df['minimum_nights'].clip(upper=30)
    
    # ── 6. Aktivitäts-Score ──────────────────────────────────────────────────
    df['review_activity_score'] = (
        df['reviews_per_month'].fillna(0) * df['availability_365']
    )
    
    # ── 7. One-Hot-Encoding: room_type & region ─────────────────────────────
    df = pd.get_dummies(df, columns=['room_type', 'region'], drop_first=False)
    
    # ── 8. Features auswählen (nur numerische, keine Leak-Variablen) ─────────
    base_features = [
        'minimum_nights', 'min_nights_capped',
        'number_of_reviews', 'reviews_per_month',
        'calculated_host_listings_count',
        'availability_365', 'number_of_reviews_ltm',
        'dist_city_center', 'dist_beach',
        'is_licensed', 'host_is_pro',
        'review_activity_score',
        'latitude', 'longitude',
    ]
    ohe_features = [c for c in df.columns if c.startswith('room_type_') or c.startswith('region_')]
    all_features  = base_features + ohe_features
    
    # Fehlende reviews_per_month mit 0 auffüllen
    df['reviews_per_month'] = df['reviews_per_month'].fillna(0)
    
    X = df[all_features].copy()
    y = df['price'].copy()
    
    print(f'Feature Engineering abgeschlossen: {X.shape[1]} Features, {X.shape[0]:,} Samples')
    print(f'Features: {all_features}')
    return X, y, all_features


X, y, FEATURE_NAMES = build_features(df_raw)
X.head(3)

## 3 – Train / Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=RANDOM_STATE
)

print(f'Trainingsdaten:  {X_train.shape[0]:,} Zeilen')
print(f'Testdaten:       {X_test.shape[0]:,} Zeilen')
print(f'Zielvariable y – Median: {y_train.median():.1f} € | Mean: {y_train.mean():.1f} €')

## 4 – Modell III: Optimierter Decision Tree

### Theoretische Begründung
Ein Decision Tree Regressor ist ein nicht-parametrisches Modell, das den Merkmalsraum
durch rekursive binäre Splits in Regionen aufteilt. Im Gegensatz zur linearen Regression
kann er nicht-lineare Zusammenhänge und Interaktionseffekte zwischen Features abbilden –
etwa zwischen Region und Zimmertyp bei der Preisbildung.

**Hyperparameter-Tuning via GridSearchCV** (5-Fold Cross-Validation):

| Parameter | Suchraum | Bedeutung |
|---|---|---|
| `max_depth` | [4, 6, 8, 10, 12] | Tiefe des Baums – kontrolliert Overfitting |
| `min_samples_split` | [20, 50, 100] | Mindestgröße für einen Split |
| `min_samples_leaf` | [10, 20, 50] | Mindestgröße eines Blatts |
| `max_features` | ['sqrt', 'log2', None] | Feature-Sampling pro Split |

In [ ]:
# ── Hyperparameter-Grid ─────────────────────────────────────────────────────
param_grid = {
    'max_depth':          [4, 6, 8, 10, 12],
    'min_samples_split':  [20, 50, 100],
    'min_samples_leaf':   [10, 20, 50],
    'max_features':       ['sqrt', 'log2', None],
}

dt_base = DecisionTreeRegressor(random_state=RANDOM_STATE)

grid_search = GridSearchCV(
    estimator=dt_base,
    param_grid=param_grid,
    cv=5,
    scoring='neg_mean_absolute_error',
    n_jobs=-1,
    verbose=1,
    refit=True
)

print('Starte GridSearchCV (Decision Tree) – kann 2-4 Minuten dauern...')
grid_search.fit(X_train, y_train)

best_dt = grid_search.best_estimator_
print(f'\nBeste Parameter: {grid_search.best_params_}')
print(f'Bestes CV-MAE:   {-grid_search.best_score_:.2f} €')

In [ ]:
# ── Evaluation auf dem Testset ──────────────────────────────────────────────
y_pred_dt = best_dt.predict(X_test)

mae_dt  = mean_absolute_error(y_test, y_pred_dt)
rmse_dt = np.sqrt(mean_squared_error(y_test, y_pred_dt))
r2_dt   = r2_score(y_test, y_pred_dt)

print('=' * 45)
print('  MODELL III – Optimierter Decision Tree')
print('=' * 45)
print(f'  MAE:   {mae_dt:.2f} €')
print(f'  RMSE:  {rmse_dt:.2f} €')
print(f'  R²:    {r2_dt:.4f}')
print('=' * 45)

## 4b – Fallback: GradientBoostingRegressor

Nur ausführen wenn R² des Decision Tree < 0.35 ist.  
Die Zelle kann dann einfach aktiviert und ausgeführt werden.

In [ ]:
# ── NUR AUSFÜHREN WENN DECISION TREE NICHT ZUFRIEDENSTELLEND ────────────────
# Bedingung: r2_dt < 0.35

USE_FALLBACK = r2_dt < 0.35   # Automatische Entscheidung

if USE_FALLBACK:
    print('Decision Tree R² unter Schwellenwert → Starte GradientBoostingRegressor...')
    
    gb_param_grid = {
        'n_estimators':   [100, 200],
        'max_depth':      [3, 4, 5],
        'learning_rate':  [0.05, 0.1, 0.2],
        'subsample':      [0.8, 1.0],
    }
    
    gb_base = GradientBoostingRegressor(random_state=RANDOM_STATE)
    
    gb_grid = GridSearchCV(
        gb_base, gb_param_grid, cv=3,
        scoring='neg_mean_absolute_error',
        n_jobs=-1, verbose=1, refit=True
    )
    gb_grid.fit(X_train, y_train)
    
    best_model_III  = gb_grid.best_estimator_
    model_III_name  = 'GradientBoostingRegressor'
    
    y_pred_final = best_model_III.predict(X_test)
    mae_final    = mean_absolute_error(y_test, y_pred_final)
    rmse_final   = np.sqrt(mean_squared_error(y_test, y_pred_final))
    r2_final     = r2_score(y_test, y_pred_final)
    
    print(f'\nFallback-Modell beste Parameter: {gb_grid.best_params_}')
    print(f'MAE: {mae_final:.2f} € | RMSE: {rmse_final:.2f} € | R²: {r2_final:.4f}')

else:
    print(f'Decision Tree R²={r2_dt:.4f} ≥ 0.35 → kein Fallback nötig ✓')
    best_model_III = best_dt
    model_III_name = 'DecisionTreeRegressor (optimiert)'
    y_pred_final = y_pred_dt
    mae_final, rmse_final, r2_final = mae_dt, rmse_dt, r2_dt

## 5 – Feature Importance

In [ ]:
importances = best_model_III.feature_importances_
feat_imp_df = pd.DataFrame({
    'feature':    FEATURE_NAMES,
    'importance': importances
}).sort_values('importance', ascending=False).reset_index(drop=True)

print('Top 15 Features:')
print(feat_imp_df.head(15).to_string(index=False))

# Speichern für Notebook 50
feat_imp_df.to_csv(f'{CACHE_DIR}40_feature_importance.csv', index=False)
print(f'\nGespeichert → {CACHE_DIR}40_feature_importance.csv')

In [ ]:
# ── Plot: Top-15 Feature Importance ────────────────────────────────────────
top15 = feat_imp_df.head(15)

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.barh(
    top15['feature'][::-1],
    top15['importance'][::-1],
    color='steelblue', edgecolor='white'
)
ax.set_xlabel('Feature Importance (Gini / Variance Reduction)', fontsize=11)
ax.set_title(f'Top-15 Feature Importances – {model_III_name}', fontsize=13, fontweight='bold')
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}40_feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()
print('Plot gespeichert ✓')

In [ ]:
# ── Plot: Predicted vs. Actual ──────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(7, 7))
ax.scatter(y_test, y_pred_final, alpha=0.15, s=10, color='steelblue')
ax.plot([0, 1000], [0, 1000], 'r--', linewidth=1.5, label='Perfekte Vorhersage')
ax.set_xlabel('Tatsächlicher Preis (€)', fontsize=11)
ax.set_ylabel('Vorhergesagter Preis (€)', fontsize=11)
ax.set_title(f'Predicted vs. Actual – {model_III_name}', fontsize=13, fontweight='bold')
ax.legend()
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}40_predicted_vs_actual.png', dpi=150, bbox_inches='tight')
plt.show()
print('Plot gespeichert ✓')

## 6 – Ergebnisse & Modell speichern (für Notebook 50)

In [ ]:
# ── Metriken als CSV speichern ───────────────────────────────────────────────
metrics_model_III = pd.DataFrame([{
    'model':      model_III_name,
    'notebook':   '40',
    'mae':        round(mae_final, 4),
    'rmse':       round(rmse_final, 4),
    'r2':         round(r2_final, 4),
    'best_params': str(grid_search.best_params_ if not USE_FALLBACK else gb_grid.best_params_)
}])

metrics_model_III.to_csv(f'{CACHE_DIR}40_metrics_model_III.csv', index=False)
print('Metriken gespeichert:')
print(metrics_model_III.to_string(index=False))

# ── Modell als Pickle speichern ──────────────────────────────────────────────
with open(f'{CACHE_DIR}40_model_III.pkl', 'wb') as f:
    pickle.dump(best_model_III, f)
print(f'\nModell gespeichert → {CACHE_DIR}40_model_III.pkl')

# ── Feature-Namen speichern ───────────────────────────────────────────────────
pd.Series(FEATURE_NAMES).to_csv(f'{CACHE_DIR}40_feature_names.csv', index=False, header=['feature'])
print(f'Feature-Namen gespeichert → {CACHE_DIR}40_feature_names.csv')

## 7 – Zusammenfassung

> Die folgende Zelle gibt eine saubere Zusammenfassung aus, die direkt
> in Notebook 50 als Grundlage für den Modellvergleich dient.

In [ ]:
print('=' * 55)
print('  NOTEBOOK 40 – ZUSAMMENFASSUNG')
print('=' * 55)
print(f'  Modell III:  {model_III_name}')
print(f'  Features:    {len(FEATURE_NAMES)}')
print(f'  Train/Test:  {X_train.shape[0]:,} / {X_test.shape[0]:,} Samples')
print()
print(f'  MAE:         {mae_final:.2f} €')
print(f'  RMSE:        {rmse_final:.2f} €')
print(f'  R²:          {r2_final:.4f}')
print()
print('  Outputs:')
print(f'    cache/40_metrics_model_III.csv')
print(f'    cache/40_feature_importance.csv')
print(f'    cache/40_model_III.pkl')
print(f'    outputs/40_feature_importance.png')
print(f'    outputs/40_predicted_vs_actual.png')
print('=' * 55)

# ── TODO-Hinweise für Notebook 50 ──────────────────────────────────────────
print()
print('TODO für Notebook 50:')
print('  [ ] MAE, RMSE, R² von Modell I (Annika) manuell eintragen')
print('  [ ] MAE, RMSE, R² von Modell II (Mareike) manuell eintragen')
print('  [ ] Metriken-CSV von Modell I & II liegen vor? → einlesen')

---
## MODULARITÄTS-HINWEIS FÜR SPÄTERE ANPASSUNG

Sobald Mareikes `cache/20_features_engineered.csv` auf GitHub vorliegt,
gibt es zwei Optionen:

**Option A – Mareikes Features nutzen (empfohlen für Konsistenz im Team):**
```python
# Ersetze Abschnitt 2 komplett durch:
df_feat = pd.read_csv('cache/20_features_engineered.csv')
TARGET  = 'price'
# Spalten anpassen – alle Nicht-Zielvariablen als X:
drop_cols = [TARGET, 'id', 'name', 'host_name', 'last_review', 'licence', ...]
X = df_feat.drop(columns=[c for c in drop_cols if c in df_feat.columns])
y = df_feat[TARGET]
FEATURE_NAMES = X.columns.tolist()
# → Ab Abschnitt 3 (Train/Test-Split) weiter wie gehabt
```

**Option B – Eigene Features behalten:**  
Nichts ändern. `build_features()` bleibt wie sie ist.

In beiden Fällen bleibt Abschnitte 3–7 unverändert.